<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/13-performance-tuning/00-reading-ui-dag-and-skew.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)

print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))


# 13.0 — Reading the UI DAG and skew: the top-down routine

Build a job with one genuinely skewed stage among several healthy
ones, then run the exact top-down routine the lesson describes: Jobs
-> stage DAG -> per-task duration quantiles -> max-vs-median verdict.
Generalizes 12.2's `worst_tasks` from spill bytes to duration.

In [ ]:
import os, sys
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("13.0-reading-ui-dag-skew")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.sql.adaptive.enabled", "false")  # stable teaching plans (Module 7/10 convention)
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## Step 1 — the Jobs tab, scripted: which job, how many stages

`recent_stages` (10.2/12.2's helper) tells you what ran. Build a
pipeline with one skewed join stage buried among healthy ones.

In [ ]:
import urllib.request, json as _json

def app_id():
    return spark.sparkContext.applicationId

def recent_stages(n=6):
    url = (f"http://localhost:4040/api/v1/applications/{app_id()}"
           f"/stages?status=complete")
    stages = _json.load(urllib.request.urlopen(url))
    for s in sorted(stages, key=lambda s: s["stageId"])[-n:]:
        print(f"stage {s['stageId']:>3}: {s['numTasks']:>4} tasks  "
              f"duration~{s.get('executorRunTime', 0):>10,}ms(sum)")

# a healthy, evenly-distributed stage
healthy = spark.range(4_000_000).withColumn("k", (col("id") % 200).cast("int"))
healthy.groupBy("k").count().collect()

# a deliberately skewed stage: one key holds 90% of the rows (7.4/10.4's setup)
skewed = spark.range(4_000_000).withColumn(
    "k", F.when(col("id") % 100 < 90, F.lit(0)).otherwise((col("id") % 100).cast("int")))
skewed.groupBy("k").count().collect()

recent_stages()

## Step 2 — per-task duration, not stage-total: build the quantile check

This is 13.0's central move: max vs MEDIAN, not max vs min, not the
stage's total. Pull every task's duration for the last few stages.

In [ ]:
def task_durations(stage_id):
    url = f"http://localhost:4040/api/v1/applications/{app_id()}/stages/{stage_id}"
    detail = _json.load(urllib.request.urlopen(url))[0]
    return sorted(t["duration"] for t in detail["tasks"].values())

def skew_verdict(stage_id):
    durs = task_durations(stage_id)
    if not durs:
        print(f"stage {stage_id}: no task data"); return
    n = len(durs)
    median = durs[n // 2]
    mx = durs[-1]
    ratio = mx / max(median, 1)
    verdict = "SKEWED (max >> median)" if ratio > 5 else "healthy (max ~ median)"
    print(f"stage {stage_id}: median={median:>6}ms  max={mx:>6}ms  "
          f"ratio={ratio:5.1f}x  -> {verdict}")

# find the stage ids for the two groupBys we just ran
url = f"http://localhost:4040/api/v1/applications/{app_id()}/stages?status=complete"
all_stages = sorted(s["stageId"] for s in _json.load(urllib.request.urlopen(url)))
for sid in all_stages[-4:]:
    skew_verdict(sid)

## Step 3 — confirm visually: `explain()` shows the SAME shuffle key either way

The plan alone can't distinguish skewed from healthy — both show one
`Exchange hashpartitioning(k, 8)`. This is exactly why 13.0 insists on
task-level metrics, not plan-reading, to catch skew.

In [ ]:
print("healthy groupBy plan:")
healthy.groupBy("k").count().explain()
print("\nskewed groupBy plan (structurally IDENTICAL):")
skewed.groupBy("k").count().explain()
# same shuffle, same plan shape -- skew is invisible here, only in task timing

## Putting the routine in one function

The full top-down check, scripted, ready to point at any stage id.

In [ ]:
def diagnose_stage(stage_id):
    print(f"--- stage {stage_id} ---")
    skew_verdict(stage_id)
    durs = task_durations(stage_id)
    n = len(durs)
    print(f"  n_tasks={n}  min={durs[0]}ms  p25={durs[n//4]}ms  "
          f"median={durs[n//2]}ms  p75={durs[3*n//4]}ms  max={durs[-1]}ms")

for sid in all_stages[-4:]:
    diagnose_stage(sid)

In [ ]:
spark.stop()

## Exercises

1. Build a THIRD job whose stage has too few shuffle partitions for
   its data volume (10.3's problem) rather than skew. Run
   `skew_verdict()` on it — does the max/median ratio flag it as
   "skewed"? Explain why or why not, and how you'd tell the two apart
   in practice (hint: absolute magnitude, not just the ratio).
2. Change the skewed job's hot-key fraction from 90% to 99% and to
   55%. How does the max/median ratio respond? At what point does
   your `ratio > 5` threshold start missing real skew?
3. Extend `diagnose_stage` to also print total shuffle bytes read/
   written for the stage (10.2's fields) alongside the timing verdict.
4. Using only `recent_stages()` (stage-level aggregates, no per-task
   data), could you have detected the skewed stage? Try it, then
   explain in your own words why stage-level metrics can hide skew
   that task-level metrics reveal.
5. Write a version of `skew_verdict` that returns a verdict string
   instead of printing, and use it to build a one-cell "does this job
   have any skewed stages?" summary over `all_stages`.

In [ ]:
# your exercise code here